In [ ]:
import os

from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_openai import OpenAI

llm = OpenAI(model="gpt-4.1-mini", temperature=0.2)

In [ ]:
PROMPT = """# ✨ 다듬은 프롬프트 (실행용)

## 📌 역할 정의

당신은 **에듀테크 기반 영어 시험 자동화 시스템을 설계하고 구현하는 AI 개발자**입니다.
Jupyter Notebook(`english_academy_project.ipynb`)에 코드를 작성하고, 프로젝트 구조 및 기능을 단계적으로 구현하세요.

## 📌 목표

모의고사(내신형) 문제를 OCR 기반으로 추출 → 구조화 → DB 저장 → 추가 콘텐츠 생성(TTS, 이미지)까지 수행하는 시스템 구축

## 📌 1. 폴더 구조 생성

다음과 같은 디렉토리 구조를 생성하세요:

```
/data
  /questions
    /chapter1
    /chapter2
  /solutions
    /chapter1
    /chapter2
```

* 문제와 해설을 각각 장(chapter) 단위로 분리

## 📌 2. 문제 OCR 처리 (Test)

### 조건

* 5페이지 분량 처리
* LLM 기반 OCR 수행

### 출력 형식 (JSON)

```json
{
  "번호": 1,
  "지문": "...",
  "선지": [
    {"번호": 1, "내용": "..."},
    {"번호": 2, "내용": "..."}
  ]
}
```

* 밑줄이 있는 경우 `<u></u>` 태그로 표시

## 📌 3. 해설 OCR 처리

### 조건

* 4페이지 분량 처리

### 출력 형식

#### (1) 정답만 있는 경우

```json
{
  "번호": 1,
  "정답": 3
}
```

#### (2) 해설 포함된 경우

```json
{
  "번호": 1,
  "지문": "...",
  "단어": {
    "abandon": "버리다"
  }
}
```

## 📌 4. DB 저장

* OCR 결과를 DB에 저장
* 비정형 데이터(JSON) 저장 가능하도록 설계
* 추천: MongoDB 또는 JSON 기반 저장 구조

## 📌 5. 시험 출제 포인트 추출

지문 분석을 통해 다음 정보 생성:

* `<문법>` → 빨간색 강조
* `<핵심 표현>` → 파란색 강조

## 📌 6. 추가 콘텐츠 생성

### (1) Test → Image

* 문제 + 해설을 하나의 이미지로 시각화
* DB에 이미지 저장

### (2) TTS 생성

* 초등학생도 이해할 수 있는 수준으로 설명 생성
* 음성 변환 후 저장

## 📌 7. 시스템 구성

### 🔹 Backend

* OCR 처리
* LLM 호출
* DB 저장
* TTS 및 이미지 생성

### 🔹 Frontend

* 파일명 → 문제 번호 기반 리스트 출력
* 문제 클릭 시:
  * 문제
  * 해설
  * 이미지
  * 음성 재생

### 🔹 Main (Notebook)

* 전체 파이프라인 실행
* 단계별 테스트 코드 포함
* 예시 데이터로 실행 가능하도록 구성

## 📌 8. 출력 요구사항

1. `README.md` 형태로 전체 시스템 설계 정리
2. `english_academy_project.ipynb` 코드 작성
3. 각 단계별 실행 가능한 코드 포함
"""

print(PROMPT)

In [ ]:
from english_academy_project import backend, ai, frontend, marketing

# 1) 폴더 구조 생성
backend.ensure_data_dirs()

# 2) DB 초기화 및 샘플 데이터 저장
db = backend.JsonDatabase()
sample_question = {
    "번호": 1,
    "지문": "다음을 읽고 알맞은 답을 고르세요.",
    "선지": [
        {"번호": 1, "내용": "선택지 A"},
        {"번호": 2, "내용": "선택지 B"}
    ]
}
sample_solution = {
    "번호": 1,
    "정답": 2,
    "단어": {
        "abandon": "버리다"
    }
}

db.insert_question(sample_question)
db.insert_solution(sample_solution)

# 3) 프론트엔드 HTML 생성
html_path = frontend.generate_html(db.load(), output_filename="index.html")

# 4) 마케팅 플랜 생성
marketing_path = marketing.save_marketing_plan()

print("데이터 디렉터리 생성 완료:", backend.DATA_DIR)
print("HTML 파일 생성:", html_path)
print("마케팅 플랜 생성:", marketing_path)
